In [10]:
# --- imports ---
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(".."))


# ============================================================
# SETTINGS - edit these values as needed
# Keep the anchor symbol last in the list.
# ============================================================
sorted_symbols_list = ["XLU", "FUTY", "VPU"]

start_date = pd.Timestamp("2026-03-25")
end_date   = pd.Timestamp("2026-06-14")

base_directory = Path("..").resolve()

input_filename ="sectors.csv"
input_path = base_directory / "historical prices" / input_filename

output_filename = f"{'_'.join(sorted_symbols_list)}.csv"
output_path = base_directory / "backtests" / output_filename


async def main():

    df = pd.read_csv(input_path, index_col=0)
    df = df[sorted_symbols_list]

    df.reset_index(names="eop date", inplace=True)
    df["eop date"] = pd.to_datetime(df["eop date"], errors="coerce")
    df = df.loc[df["eop date"].between(start_date, end_date)].copy()
    print(df)


    df.insert(0, "bop date", df["eop date"].shift(1))

    for symbol in sorted_symbols_list:
        bop_price = f"bop {symbol} price"
        eop_price = f"eop {symbol} price"

        df[bop_price] = df[symbol].shift(1)
        df[eop_price] = df[symbol]
        df[f"{symbol} cop"] = df[eop_price] - df[bop_price]
        df[f"{symbol} pct cop"] = np.log(df[eop_price] / df[bop_price])
        df.drop(columns=symbol, inplace=True)

    df.to_csv(output_path)
    print(f"Saved {output_path}")
    print("finished")


await main()


       eop date    XLU   FUTY     VPU
1146 2026-03-25  45.25  58.34  195.48
1147 2026-03-26  45.33  58.47  195.92
1148 2026-03-27  45.59  58.74  196.88
1149 2026-03-30  45.92  59.13  198.11
1150 2026-03-31  45.89  59.07  198.14
1151 2026-04-01  46.11  59.36  198.97
1152 2026-04-02  46.34  59.75  200.14
1153 2026-04-06  46.17  59.48  199.40
1154 2026-04-07  46.27  59.67  199.98
1155 2026-04-08  46.78  60.30  202.09
1156 2026-04-09  47.15  60.76  203.63
1157 2026-04-10  46.96  60.52  202.89
1158 2026-04-13  46.39  59.76  200.25
1159 2026-04-14  46.47  59.96  200.91
1160 2026-04-15  46.02  59.46  199.25
1161 2026-04-16  46.35  59.85  200.51
1162 2026-04-17  46.16  59.59  199.70
1163 2026-04-20  45.75  59.11  198.05
1164 2026-04-21  44.95  58.04  194.44
1165 2026-04-22  44.87  57.97  194.34
1166 2026-04-23  46.09  59.59  199.69
1167 2026-04-24  46.18  59.65  199.84
1168 2026-04-27  46.19  59.72  200.11
1169 2026-04-28  46.25  59.75  200.26
1170 2026-04-29  45.68  59.00  197.60
1171 2026-04